# Retail Analytics Project

**Problem Statement:** A nationwide retail chain receives daily sales data from all branches. The data engineering team loads CSV files into Snowflake, and the BI team generates analytical reports to identify top-performing products, branches, and customers.

---
## PHASE 1: ENVIRONMENT SETUP

In [ ]:
%%sql -r setup_env
USE DATABASE RETAIL_DB;
USE SCHEMA SALES_SCHEMA;
USE WAREHOUSE COMPUTE_WH;

---
## PHASE 2: DATA LOADING
### 2.1 Verify CSV Files on Internal Stage

In [ ]:
%%sql -r stage_files
LIST @RETAIL_STAGE;

### 2.2 Create Tables with Appropriate Data Types

In [ ]:
%%sql -r create_customers
CREATE OR REPLACE TABLE CUSTOMERS (
    CUSTOMER_ID INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    CITY VARCHAR(50),
    MEMBERSHIP VARCHAR(20)
);

In [ ]:
%%sql -r create_products
CREATE OR REPLACE TABLE PRODUCTS (
    PRODUCT_ID INT PRIMARY KEY,
    PRODUCT_NAME VARCHAR(100),
    CATEGORY VARCHAR(50),
    PRICE DECIMAL(10,2)
);

In [ ]:
%%sql -r create_branches
CREATE OR REPLACE TABLE BRANCHES (
    BRANCH_ID INT PRIMARY KEY,
    BRANCH_NAME VARCHAR(100),
    CITY VARCHAR(50)
);

In [ ]:
%%sql -r create_sales
CREATE OR REPLACE TABLE SALES (
    SALE_ID INT PRIMARY KEY,
    CUSTOMER_ID INT,
    PRODUCT_ID INT,
    BRANCH_ID INT,
    QUANTITY INT,
    SALE_DATE DATE,
    TOTAL_AMOUNT DECIMAL(10,2),
    FOREIGN KEY (CUSTOMER_ID) REFERENCES CUSTOMERS(CUSTOMER_ID),
    FOREIGN KEY (PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID),
    FOREIGN KEY (BRANCH_ID) REFERENCES BRANCHES(BRANCH_ID)
);

### 2.3 Load CSV Files Using COPY INTO

In [ ]:
%%sql -r load_customers
COPY INTO CUSTOMERS
FROM @RETAIL_STAGE/customers.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

In [ ]:
%%sql -r load_products
COPY INTO PRODUCTS
FROM @RETAIL_STAGE/products.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

In [ ]:
%%sql -r load_branches
COPY INTO BRANCHES
FROM @RETAIL_STAGE/branches.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

In [ ]:
%%sql -r load_sales
COPY INTO SALES
FROM @RETAIL_STAGE/sales.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

### 2.4 Verify Imported Records

In [ ]:
%%sql -r row_counts
SELECT 'CUSTOMERS' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM CUSTOMERS
UNION ALL
SELECT 'PRODUCTS', COUNT(*) FROM PRODUCTS
UNION ALL
SELECT 'BRANCHES', COUNT(*) FROM BRANCHES
UNION ALL
SELECT 'SALES', COUNT(*) FROM SALES;

---
## PHASE 3: SQL ANALYTICS
### 3.1 Basic Data Display

In [ ]:
%%sql -r all_customers
-- Display all customers
SELECT * FROM CUSTOMERS;

In [ ]:
%%sql -r all_products
-- Display all products
SELECT * FROM PRODUCTS;

In [ ]:
%%sql -r all_branches
-- Display all branches
SELECT * FROM BRANCHES;

In [ ]:
%%sql -r all_sales
-- Display all sales transactions
SELECT * FROM SALES;

### 3.2 Aggregate Functions - Revenue Analysis

In [ ]:
%%sql -r total_revenue
-- Total business revenue
SELECT SUM(TOTAL_AMOUNT) AS TOTAL_REVENUE FROM SALES;

In [ ]:
%%sql -r customer_sales
-- Customer-wise sales
SELECT 
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.CITY,
    c.MEMBERSHIP,
    COUNT(s.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_SPENDING
FROM CUSTOMERS c
JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
GROUP BY c.CUSTOMER_ID, c.CUSTOMER_NAME, c.CITY, c.MEMBERSHIP
ORDER BY TOTAL_SPENDING DESC;

In [ ]:
%%sql -r branch_sales
-- Branch-wise sales
SELECT 
    b.BRANCH_ID,
    b.BRANCH_NAME,
    b.CITY,
    COUNT(s.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM BRANCHES b
JOIN SALES s ON b.BRANCH_ID = s.BRANCH_ID
GROUP BY b.BRANCH_ID, b.BRANCH_NAME, b.CITY
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r product_sales
-- Product-wise sales
SELECT 
    p.PRODUCT_ID,
    p.PRODUCT_NAME,
    p.CATEGORY,
    p.PRICE AS UNIT_PRICE,
    SUM(s.QUANTITY) AS TOTAL_QTY_SOLD,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM PRODUCTS p
JOIN SALES s ON p.PRODUCT_ID = s.PRODUCT_ID
GROUP BY p.PRODUCT_ID, p.PRODUCT_NAME, p.CATEGORY, p.PRICE
ORDER BY TOTAL_REVENUE DESC;

In [ ]:
%%sql -r category_sales
-- Category-wise sales
SELECT 
    p.CATEGORY,
    COUNT(s.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(s.QUANTITY) AS TOTAL_QTY_SOLD,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM PRODUCTS p
JOIN SALES s ON p.PRODUCT_ID = s.PRODUCT_ID
GROUP BY p.CATEGORY
ORDER BY TOTAL_REVENUE DESC;

### 3.3 Top Performers

In [ ]:
%%sql -r top_branch
-- Highest revenue branch
SELECT 
    b.BRANCH_NAME,
    b.CITY,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM BRANCHES b
JOIN SALES s ON b.BRANCH_ID = s.BRANCH_ID
GROUP BY b.BRANCH_NAME, b.CITY
ORDER BY TOTAL_REVENUE DESC
LIMIT 1;

In [ ]:
%%sql -r top_customer
-- Highest spending customer
SELECT 
    c.CUSTOMER_NAME,
    c.CITY,
    c.MEMBERSHIP,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_SPENDING
FROM CUSTOMERS c
JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
GROUP BY c.CUSTOMER_NAME, c.CITY, c.MEMBERSHIP
ORDER BY TOTAL_SPENDING DESC
LIMIT 1;

In [ ]:
%%sql -r top3_products
-- Top 3 products by revenue
SELECT 
    p.PRODUCT_NAME,
    p.CATEGORY,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM PRODUCTS p
JOIN SALES s ON p.PRODUCT_ID = s.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY
ORDER BY TOTAL_REVENUE DESC
LIMIT 3;

In [ ]:
%%sql -r top3_customers
-- Top 3 customers by spending
SELECT 
    c.CUSTOMER_NAME,
    c.CITY,
    c.MEMBERSHIP,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_SPENDING
FROM CUSTOMERS c
JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
GROUP BY c.CUSTOMER_NAME, c.CITY, c.MEMBERSHIP
ORDER BY TOTAL_SPENDING DESC
LIMIT 3;

---
## PHASE 4: ADVANCED SQL
### 4.1 Window Functions

In [ ]:
%%sql -r customer_ranking
-- Rank customers by spending using Window Functions
SELECT 
    c.CUSTOMER_NAME,
    c.CITY,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_SPENDING,
    RANK() OVER (ORDER BY SUM(s.TOTAL_AMOUNT) DESC) AS SPENDING_RANK,
    DENSE_RANK() OVER (ORDER BY SUM(s.TOTAL_AMOUNT) DESC) AS DENSE_SPENDING_RANK
FROM CUSTOMERS c
JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
GROUP BY c.CUSTOMER_NAME, c.CITY;

In [ ]:
%%sql -r running_totals
-- Running total of sales by date with Window Function
SELECT 
    SALE_DATE,
    TOTAL_AMOUNT,
    SUM(TOTAL_AMOUNT) OVER (ORDER BY SALE_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS RUNNING_TOTAL,
    AVG(TOTAL_AMOUNT) OVER (ORDER BY SALE_DATE ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS MOVING_AVG_3DAY
FROM SALES
ORDER BY SALE_DATE;

In [ ]:
%%sql -r product_contribution
-- Product revenue contribution percentage using Window Function
SELECT 
    p.PRODUCT_NAME,
    p.CATEGORY,
    SUM(s.TOTAL_AMOUNT) AS PRODUCT_REVENUE,
    ROUND(SUM(s.TOTAL_AMOUNT) * 100.0 / SUM(SUM(s.TOTAL_AMOUNT)) OVER (), 2) AS REVENUE_PCT,
    ROW_NUMBER() OVER (PARTITION BY p.CATEGORY ORDER BY SUM(s.TOTAL_AMOUNT) DESC) AS RANK_IN_CATEGORY
FROM PRODUCTS p
JOIN SALES s ON p.PRODUCT_ID = s.PRODUCT_ID
GROUP BY p.PRODUCT_NAME, p.CATEGORY
ORDER BY p.CATEGORY, RANK_IN_CATEGORY;

### 4.2 Common Table Expressions (CTEs)

In [ ]:
%%sql -r high_value_customers
-- CTE: Identify high-value customers (above average spending)
WITH customer_spending AS (
    SELECT 
        c.CUSTOMER_ID,
        c.CUSTOMER_NAME,
        c.MEMBERSHIP,
        SUM(s.TOTAL_AMOUNT) AS TOTAL_SPENDING
    FROM CUSTOMERS c
    JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
    GROUP BY c.CUSTOMER_ID, c.CUSTOMER_NAME, c.MEMBERSHIP
),
avg_spending AS (
    SELECT AVG(TOTAL_SPENDING) AS AVG_SPEND FROM customer_spending
)
SELECT 
    cs.CUSTOMER_NAME,
    cs.MEMBERSHIP,
    cs.TOTAL_SPENDING,
    a.AVG_SPEND,
    CASE WHEN cs.TOTAL_SPENDING > a.AVG_SPEND THEN 'HIGH VALUE' ELSE 'REGULAR' END AS CUSTOMER_SEGMENT
FROM customer_spending cs
CROSS JOIN avg_spending a
ORDER BY cs.TOTAL_SPENDING DESC;

In [ ]:
%%sql -r branch_category_breakdown
-- CTE: Branch performance with category breakdown
WITH branch_category_sales AS (
    SELECT 
        b.BRANCH_NAME,
        p.CATEGORY,
        SUM(s.TOTAL_AMOUNT) AS CATEGORY_REVENUE
    FROM SALES s
    JOIN BRANCHES b ON s.BRANCH_ID = b.BRANCH_ID
    JOIN PRODUCTS p ON s.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY b.BRANCH_NAME, p.CATEGORY
),
branch_totals AS (
    SELECT 
        BRANCH_NAME,
        SUM(CATEGORY_REVENUE) AS TOTAL_BRANCH_REVENUE
    FROM branch_category_sales
    GROUP BY BRANCH_NAME
)
SELECT 
    bcs.BRANCH_NAME,
    bcs.CATEGORY,
    bcs.CATEGORY_REVENUE,
    bt.TOTAL_BRANCH_REVENUE,
    ROUND(bcs.CATEGORY_REVENUE * 100.0 / bt.TOTAL_BRANCH_REVENUE, 2) AS PCT_OF_BRANCH
FROM branch_category_sales bcs
JOIN branch_totals bt ON bcs.BRANCH_NAME = bt.BRANCH_NAME
ORDER BY bcs.BRANCH_NAME, bcs.CATEGORY_REVENUE DESC;

### 4.3 Views and Materialized Views

In [ ]:
%%sql -r create_view
-- Create a comprehensive Sales Report View (multi-table join)
CREATE OR REPLACE VIEW SALES_REPORT_VIEW AS
SELECT 
    s.SALE_ID,
    s.SALE_DATE,
    c.CUSTOMER_NAME,
    c.CITY AS CUSTOMER_CITY,
    c.MEMBERSHIP,
    p.PRODUCT_NAME,
    p.CATEGORY,
    p.PRICE AS UNIT_PRICE,
    b.BRANCH_NAME,
    b.CITY AS BRANCH_CITY,
    s.QUANTITY,
    s.TOTAL_AMOUNT
FROM SALES s
JOIN CUSTOMERS c ON s.CUSTOMER_ID = c.CUSTOMER_ID
JOIN PRODUCTS p ON s.PRODUCT_ID = p.PRODUCT_ID
JOIN BRANCHES b ON s.BRANCH_ID = b.BRANCH_ID;

In [ ]:
%%sql -r view_data
-- Query the Sales Report View
SELECT * FROM SALES_REPORT_VIEW
ORDER BY SALE_DATE, SALE_ID;

In [ ]:
%%sql -r create_mv
-- Create a Materialized View for Sales Summary by Date
-- Note: Snowflake MVs support single-table references only
CREATE OR REPLACE MATERIALIZED VIEW MV_DAILY_SALES_SUMMARY AS
SELECT 
    SALE_DATE,
    COUNT(SALE_ID) AS NUM_TRANSACTIONS,
    SUM(QUANTITY) AS TOTAL_ITEMS_SOLD,
    SUM(TOTAL_AMOUNT) AS DAILY_REVENUE,
    AVG(TOTAL_AMOUNT) AS AVG_ORDER_VALUE,
    MAX(TOTAL_AMOUNT) AS MAX_ORDER,
    MIN(TOTAL_AMOUNT) AS MIN_ORDER
FROM SALES
GROUP BY SALE_DATE;

In [ ]:
%%sql -r mv_data
-- Query the Materialized View
SELECT * FROM MV_DAILY_SALES_SUMMARY
ORDER BY SALE_DATE;

---
## PHASE 5: BUSINESS INTELLIGENCE REPORTS

In [ ]:
%%sql -r dashboard_summary
-- BI Report: Complete Sales Dashboard Summary
SELECT 
    COUNT(DISTINCT s.SALE_ID) AS TOTAL_TRANSACTIONS,
    COUNT(DISTINCT s.CUSTOMER_ID) AS ACTIVE_CUSTOMERS,
    COUNT(DISTINCT s.PRODUCT_ID) AS PRODUCTS_SOLD,
    COUNT(DISTINCT s.BRANCH_ID) AS ACTIVE_BRANCHES,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE,
    AVG(s.TOTAL_AMOUNT) AS AVG_ORDER_VALUE,
    MAX(s.TOTAL_AMOUNT) AS LARGEST_ORDER,
    MIN(s.SALE_DATE) AS FIRST_SALE_DATE,
    MAX(s.SALE_DATE) AS LAST_SALE_DATE
FROM SALES s;

In [ ]:
%%sql -r daily_trend
-- BI Report: Daily Sales Trend
SELECT 
    SALE_DATE,
    COUNT(SALE_ID) AS NUM_TRANSACTIONS,
    SUM(TOTAL_AMOUNT) AS DAILY_REVENUE,
    SUM(SUM(TOTAL_AMOUNT)) OVER (ORDER BY SALE_DATE) AS CUMULATIVE_REVENUE
FROM SALES
GROUP BY SALE_DATE
ORDER BY SALE_DATE;

In [ ]:
%%sql -r membership_analysis
-- BI Report: Customer Purchasing Behavior by Membership
SELECT 
    c.MEMBERSHIP,
    COUNT(DISTINCT c.CUSTOMER_ID) AS NUM_CUSTOMERS,
    COUNT(s.SALE_ID) AS TOTAL_TRANSACTIONS,
    SUM(s.TOTAL_AMOUNT) AS TOTAL_REVENUE,
    ROUND(SUM(s.TOTAL_AMOUNT) / COUNT(DISTINCT c.CUSTOMER_ID), 2) AS AVG_REVENUE_PER_CUSTOMER,
    ROUND(AVG(s.TOTAL_AMOUNT), 2) AS AVG_ORDER_VALUE
FROM CUSTOMERS c
JOIN SALES s ON c.CUSTOMER_ID = s.CUSTOMER_ID
GROUP BY c.MEMBERSHIP
ORDER BY TOTAL_REVENUE DESC;